# 09-3. GGUF 변환 · Llama.cpp 추론

**입력은 02 의 GPTQ 모델이 아니라 01 의 merged fp16 모델**

llama.cpp 는 GPTQ/bnb 가중치를 읽지 못함. GGUF 의 K-quant 는 그 자체가 독립적인 PTQ 경로라서,
02 와 03 은 *같은 fp16 원본에서 갈라지는 두 가지 양자화 방식* 으로 비교하는 게 맞음

```
merged fp16 ─┬─ [02] GPTQ / bnb  → GPU 서빙용
             └─ [03] GGUF Q4_K_M → CPU · 로컬 실행용
```

**런타임: CPU 로도 충분** (1.5B 모델 기준)

## 0. llama.cpp 빌드

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DGGML_NATIVE=ON -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --config Release -j$(nproc)
!ls llama.cpp/build/bin | head

In [ ]:
# 변환 스크립트 의존성 (requirements.txt 는 torch 를 재설치하므로 gguf 만 설치)
!pip install -q gguf sentencepiece

In [ ]:
import os, json, subprocess
HF_USER     = "본인_HF_아이디"          # TODO
MERGED_REPO = f"{HF_USER}/qwen2.5-1.5b-koalpaca-merged"
SAVE_DIR    = "/content/drive/MyDrive/ktb/09_llm_opt"
BIN         = "llama.cpp/build/bin"

from google.colab import drive, userdata
drive.mount('/content/drive')
from huggingface_hub import login, snapshot_download
login(userdata.get('HF_TOKEN'))

snapshot_download(MERGED_REPO, local_dir="/content/merged")
!ls -la /content/merged

## 1. HF → GGUF 변환 (f16)

`convert_hf_to_gguf.py` 가 safetensors 가중치 + config + tokenizer 를
**단일 바이너리 파일 하나로 재패킹**함. 이 단계에서는 아직 양자화하지 않음 (f16 유지)

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py /content/merged \
    --outfile /content/qwen-merged-f16.gguf --outtype f16
!ls -lh /content/*.gguf

## 2. GGUF 양자화

`llama-quantize` 로 K-quant 적용

| 타입 | 설명 |
|---|---|
| `Q8_0` | 8bit, 품질 손실 거의 없음 |
| `Q5_K_M` | 5bit 혼합, 품질·용량 균형 |
| `Q4_K_M` | 4bit 혼합, **가장 널리 쓰이는 실전 기본값** |

> `_K` = K-quant (블록 단위 super-block scale), `_M` = medium — attention 등 민감한 레이어는
> 더 높은 비트를 남겨두는 **혼합 정밀도** 전략

In [ ]:
for q in ["Q8_0", "Q5_K_M", "Q4_K_M"]:
    !{BIN}/llama-quantize /content/qwen-merged-f16.gguf /content/qwen-{q}.gguf {q}
!ls -lh /content/*.gguf

## 3. Llama.cpp 추론

In [ ]:
!{BIN}/llama-cli -m /content/qwen-Q4_K_M.gguf \
    -p "머신러닝과 딥러닝의 차이를 설명해줘" \
    -n 200 -no-cnv --temp 0

`-no-cnv` 는 대화 모드를 끄고 단발 생성. Qwen chat template 은 GGUF 메타데이터에 함께 들어가 자동 적용됨

## 4. 벤치마크 · 결과 정리

In [ ]:
!{BIN}/llama-bench -m /content/qwen-merged-f16.gguf \
                   -m /content/qwen-Q8_0.gguf \
                   -m /content/qwen-Q5_K_M.gguf \
                   -m /content/qwen-Q4_K_M.gguf -p 128 -n 128

In [ ]:
import pandas as pd, glob
rows = []
for p in sorted(glob.glob("/content/qwen-*.gguf")):
    rows.append({"file": os.path.basename(p),
                 "size_GB": round(os.path.getsize(p) / 1024**3, 2)})
df = pd.DataFrame(rows)
df["ratio"] = (df.size_GB / df.size_GB.max()).round(2)
df.to_csv(f"{SAVE_DIR}/metrics_gguf.csv", index=False)
df

In [ ]:
# Q4_K_M 만 Drive 로 저장 (로컬 Mac 에서 실행해보려면 여기서 받기)
!cp /content/qwen-Q4_K_M.gguf "{SAVE_DIR}"/
!ls -lh "{SAVE_DIR}"

## 5. 로컬(Apple Silicon) 실행

Drive 에서 `.gguf` 를 내려받은 뒤 맥에서 실행하면 Metal 백엔드로 GPU 가속됨

```bash
brew install llama.cpp
llama-cli -m qwen-Q4_K_M.gguf -p "안녕?" -n 200 -no-cnv
llama-server -m qwen-Q4_K_M.gguf --port 8080   # OpenAI 호환 API
```

**정리 포인트** — 3단계를 관통하는 흐름
1. QLoRA: 4bit 양자화로 **학습 비용**을 낮춤
2. PTQ: 양자화로 **서빙 메모리**를 낮춤
3. GGUF: 양자화된 모델을 **CPU·로컬에서 실행 가능한 포맷**으로 재패킹

같은 "양자화"라도 목적이 학습 / 서빙 / 배포로 각각 다름